In [1]:
!pip install -q numpy==2.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 59.7 MB/s eta 0:00:00


In [2]:
!pip install -q \
langchain-text-splitters \
langchain-community \
langchain-huggingface \
sentence-transformers \
faiss-cpu \
deepeval \
groq \
scikit-learn>=1.6

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
import os

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

from groq import Groq

In [4]:
from dotenv import load_dotenv

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

client = Groq()

def llm(prompt):
    res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
    )
    return res.choices[0].message.content

In [5]:
text = """
AMOLED (active-matrix organic light-emitting diode) is a type of OLED display device technology. OLED describes a specific type of thin-film-display technology in which organic compounds form the electroluminescent material, and active matrix refers to the technology behind the addressing of pixels.
An AMOLED display consists of an active matrix of OLED pixels generating light (luminescence) upon electrical activation that have been deposited or integrated onto a thin-film transistor (TFT) array, which functions as a series of switches to control the current flowing to each individual pixel.
Typically, this continuous current flow is controlled by at least two TFTs at each pixel (to trigger the luminescence), with one TFT to start and stop the charging of a storage capacitor and the second to provide a voltage source at the level needed to create a constant current to the pixel, thereby eliminating the need for the very high currents required for passive-matrix OLED operation.
TFT backplane technology is crucial in the fabrication of AMOLED displays. In AMOLEDs, the two primary TFT backplane technologies, polycrystalline silicon (poly-Si) and amorphous silicon (a-Si), are currently used offering the potential for directly fabricating the active-matrix backplanes at low temperatures (below 150 °C) onto flexible plastic substrates for producing flexible AMOLED displays.
Since 2007, AMOLED displays have been used in mobile phones, media players, TVs, and digital cameras. The current progress for this technology is towards lower power usage, lower cost, and higher screen resolutions (e.g., 8K).
Manufacturers have developed in-cell touch panels, integrating the production of capacitive sensor arrays in the AMOLED module fabrication process. Researchers at DuPont used computational fluid dynamics (CFD) software to optimize coating processes for a new solution-coated AMOLED display technology that is competitive in cost and performance with existing chemical vapor deposition (CVD) technology. Using custom modeling and analytic approaches, Samsung has developed short and long-range film-thickness control and uniformity that is commercially viable at large glass sizes.
AMOLED displays are proved to be better at providing higher refresh rates than those of passive-matrix,often have response times less than a millisecond, and they consume significantly less power.This advantage makes active-matrix OLEDs well-suited for portable electronics due to its high productivity for everyday use. AMOLED also stands higher in the field of less power consumer than OLED, because "each pixel have their own light and can be controlled leading to better power control and amplification", where power consumption is critical to battery life.
Schematic of an active-matrix OLED display
The amount of power the display consumes varies significantly depending on the color and brightness shown. As an example, one old OLED display consumes 0.3 watts while showing white text on a black background, but more than 0.7 watts showing black text on a white background, while an LCD may consume only a constant 0.35 watts regardless of what is being shown on screen. A new FHD+ or WQHD+ display will consume much more. Because the black pixels turn completely off, AMOLED also has contrast ratios that are significantly higher than LCDs.

AMOLED displays are often difficult to see in direct sunlight compared with LCDs because of their reduced maximum brightness.[19] Super AMOLED, a modern technology, addresses this issue by reducing the size of gaps between layers of the screen.[20][21] Additionally, PenTile technology is often used for a higher resolution display while requiring fewer subpixels than needed otherwise, sometimes resulting in a display less sharp and more grainy than a non-PenTile display with the same resolution.[22] The organic materials used in AMOLED displays are very prone to degradation over a relatively short period of time, resulting in color shifts as one color fades faster than another, image persistence, or burn-in
Super AMOLED is a marketing term created by Samsung for an AMOLED display where the touch screen digitizer, the layer that detects touch, is integrated into the display and cannot be separated from the display itself, rather than being overlaid on top of it. When compared with a regular LCD, a regular AMOLED display consumes less power, provides more vivid picture quality, and renders faster motion response.  However, Super AMOLED is even better at this with 20% brighter screen, 20% lower power consumption and 80% less sunlight reflection. According to Samsung, Super AMOLED reflects one-fifth as much sunlight as the first generation AMOLED. The generic term for this technology is One Glass Solution (OGS), a touchscreen technology that combines the touch sensor and cover glass into a single layer, reducing overall thickness and improving optical clarity. This is achieved by coating and etching the ITO (Indium Tin Oxide) layer directly onto the cover glass, eliminating the need for a separate sensor glass and an air gap.
Super AMOLED displays, while known for their vivid colors and deep blacks, also have some drawbacks, including higher manufacturing costs, potential for screen burn-in, and shorter lifespan compared to some other technologies.
Applications of AMOLED Displays:
AMOLED displays are widely used in smartphones, smartwatches, televisions, and virtual reality headsets. Companies like Samsung and Apple use AMOLED panels in their flagship devices due to their superior color reproduction and energy efficiency. Flexible AMOLED displays are also used in foldable phones.

Comparison with LCD:
Unlike LCD displays, AMOLED does not require a backlight. Each pixel emits its own light, allowing true blacks and higher contrast ratios. However, LCDs generally offer better brightness levels and longer lifespan.

Manufacturing Challenges:
AMOLED production is complex and expensive due to the use of organic materials and precise deposition techniques. Issues like uniformity, defect rates, and material degradation increase manufacturing costs.

Future Developments:
Research is focused on improving lifespan, reducing burn-in, and enhancing brightness. Flexible and transparent displays are also being explored for future applications in wearable devices and augmented reality systems.
"""

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = splitter.split_text(text)

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_texts(chunks, embedding)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
def rag(question):
    docs = retriever.invoke(question)
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
You are a strict RAG system.

Rules:
- Use ONLY the provided context
- If the answer is NOT in the context, say:
  "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

    answer = llm(prompt)
    return answer.strip(), context

In [8]:
def evaluate(question, answer, context):
    prompt = f"""
Evaluate this answer.

Question: {question}
Answer: {answer}
Context: {context}

Score the following from 0 to 1:
- Faithfulness (is answer grounded in context?)
- Relevancy (does it answer the question?)

Respond ONLY in this format:
Faithfulness: <score>
Relevancy: <score>
Verdict: PASS or FAIL
"""

    res = llm(prompt)

    # simple parsing
    try:
        lines = res.split("\n")
        f = float(lines[0].split(":")[1].strip())
        r = float(lines[1].split(":")[1].strip())
        verdict = lines[2].split(":")[1].strip()
    except:
        f, r, verdict = 0.0, 0.0, "FAIL"

    return verdict, f, r

In [9]:
def revise(question, answer, context):
    prompt = f"""
The previous answer was poor.

Fix it using ONLY the context.

If answer is not in context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Bad Answer:
{answer}

Improved Answer:
"""

    return llm(prompt).strip()

In [10]:
import os
os.environ["OPENAI_API_KEY"] = "sk-dummy"

In [11]:
def pipeline(q):
    ans, ctx = rag(q)

    verdict, f1, r1 = evaluate(q, ans, ctx)

    if verdict == "FAIL":
        new_ans = revise(q, ans, ctx)
        verdict2, f2, r2 = evaluate(q, new_ans, ctx)

        return {
            "question": q,
            "initial_answer": ans,
            "initial_faith": f1,
            "initial_rel": r1,
            "verdict": verdict,
            "final_answer": new_ans,
            "final_faith": f2,
            "final_rel": r2
        }
    else:
        return {
            "question": q,
            "initial_answer": ans,
            "initial_faith": f1,
            "initial_rel": r1,
            "verdict": verdict,
            "final_answer": ans,
            "final_faith": f1,
            "final_rel": r1
        }

In [12]:
questions = [
    "What is AMOLED?",
    "Compare LCD and AMOLED",
    "What is a SAMOLED?",
    "What are the advantages of AMOLED display?",
    "What is quantum computing?",
    "Give uses of AMOLED"
]

results = [pipeline(q) for q in questions]
for r in results:
    print("QUESTION:", r["question"])
    print("INITIAL:", r["initial_answer"])
    print("FINAL:", r["final_answer"])
    print("SCORES:", r["initial_faith"], r["initial_rel"], "→", r["final_faith"], r["final_rel"])
    print("VERDICT:", r["verdict"])
    print("-" * 60)

QUESTION: What is AMOLED?
INITIAL: AMOLED (active-matrix organic light-emitting diode) is a type of OLED display device technology.
FINAL: AMOLED (active-matrix organic light-emitting diode) is a type of OLED display device technology.
SCORES: 1.0 1.0 → 1.0 1.0
VERDICT: PASS
------------------------------------------------------------
QUESTION: Compare LCD and AMOLED
INITIAL: AMOLED displays are often difficult to see in direct sunlight compared with LCDs because of their reduced maximum brightness.
FINAL: AMOLED displays are often difficult to see in direct sunlight compared with LCDs because of their reduced maximum brightness. However, in other aspects, AMOLED displays consume less power, provide more vivid picture quality, and render faster motion response compared to LCDs.
SCORES: 1.0 0.5 → 0.8 0.9
VERDICT: FAIL
------------------------------------------------------------
QUESTION: What is a SAMOLED?
INITIAL: Super AMOLED is a marketing term created by Samsung for an AMOLED displa

In [13]:
import pandas as pd

df = pd.DataFrame(results)

df.columns = [
    "Question",
    "Initial Answer",
    "Final Answer",
    "Initial Faith",
    "Initial Relevancy",
    "Verdict",
    "Final Faith",
    "Final Relevancy"
]

df

,Question,Initial Answer,Final Answer,Initial Faith,Initial Relevancy,Verdict,Final Faith,Final Relevancy
0,What is AMOLED?,AMOLED (active-matrix organic light-emitting d...,1.0,1.0,PASS,AMOLED (active-matrix organic light-emitting d...,1.0,1.0
1,Compare LCD and AMOLED,AMOLED displays are often difficult to see in ...,1.0,0.5,FAIL,AMOLED displays are often difficult to see in ...,0.8,0.9
2,What is a SAMOLED?,Super AMOLED is a marketing term created by Sa...,1.0,1.0,PASS,Super AMOLED is a marketing term created by Sa...,1.0,1.0
3,What are the advantages of AMOLED display?,I don't know based on the provided context.,0.0,0.0,FAIL,The context doesn't explicitly state the advan...,1.0,0.5
4,What is quantum computing?,I don't know based on the provided context.,1.0,0.0,FAIL,I don't know based on the provided context.,1.0,0.0
5,Give uses of AMOLED,I don't know based on the provided context.,0.0,0.0,FAIL,The provided context does not explicitly menti...,1.0,0.0
